# Evaluasi terkontrol: baseline epoch864 vs V10.11 epoch868
Run all di Colab GPU. Tidak ada training atau perubahan checkpoint.
Wajib models/best_v1010_stage2_faa0e6fc16.pth (epoch864) dan models/checkpoint_v1011_067bc9a7ae.pth (epoch868). Epoch868 yang diuji adalah V10.11, bukan V10.10.
Benchmark dimatikan, cuDNN deterministic diaktifkan, dan TF32 dimatikan sebelum model digunakan. Ini menerapkan protokol yang stabil pada audit sebelumnya; bukan jaminan identik lintas perangkat.
Enam video validation yang sama, dua pengulangan penuh tiap checkpoint (termasuk pembuatan ulang frame watermark dan kompresi). Kondisi: identity, H.264/H.265 CRF22 dan CRF26, Neural Q3. Enam video tetap enam sampel, bukan dua belas.
Hasil tersimpan per video. Run ulang melanjutkan pekerjaan jika terputus. Metadata sesi dan runtime dicatat.
ZIP diunduh setelah selesai. Kirim ZIP untuk membandingkan recovery, PSNR, target stage, dan konsistensi pengulangan. Jangan tambah training sebelum hasil ini ditinjau.


Evaluasi ulang menggunakan pengaturan matematika terkontrol.


Evaluasi ulang menggunakan pengaturan matematika terkontrol.


In [ ]:
# 1. Setup Colab
from google.colab import drive
drive.mount('/content/drive')

!pip install -q kagglehub opencv-python-headless scikit-image pandas matplotlib tqdm compressai pytorch-msssim==1.0.0 "gradio>=4.44,<7"
!apt-get -qq update
!apt-get -qq install -y ffmpeg

print('Dependency siap.')


In [ ]:
# 2. Import dan konfigurasi eksperimen
import binascii
import copy
import gc
import hashlib
import json
import math
import os
import random
import re
import struct
import subprocess
import tempfile
import time
from collections import defaultdict
from pathlib import Path

import cv2
import gradio as gr
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from pytorch_msssim import ssim as differentiable_ssim
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from tqdm.auto import tqdm

SEED = 42
FRAME_SIZE = (128, 128)
CLIP_FRAMES = 16
PAYLOADS_PER_STEP = 1
MAX_EVAL_FRAMES = 64

# Profil aman untuk Colab dengan runtime terbatas. Nilai ini hanya membatasi satu
# sesi, bukan mengubah total curriculum/checkpoint penelitian.
RUNTIME_PROFILE = 'colab_limited'
CODEC_BATCH_SIZE = 4
MAX_TRAINING_EPOCHS_PER_RUN = 2
MAX_TRAINING_MINUTES_PER_RUN = 45
REUSE_EXISTING_EVAL_BITSTREAMS = True
ALLOW_DIAGNOSTIC_EVALUATION = False
RUN_MODE = 'evaluate'  # 'evaluate': muat best checkpoint tanpa training
WARM_START_CHECKPOINT = 'best_v109_stage2_ace33665af.pth'
EARLY_STOP_ROUNDS = 4
EARLY_STOP_MIN_DELTA = 0.0001  # Normalized gate deficit; cumulative since last significant improvement.
VALIDATION_VIDEOS_PER_CODEC = 6
CHECKPOINT_EVERY_STEPS = 5

PAYLOAD_BYTES = 6
PAYLOAD_ALPHABET = 'abcdefghijklmnopqrstuvwxyz012345'
PAYLOAD_BITS_PER_CHAR = 5
PAYLOAD_BITS = PAYLOAD_BYTES * PAYLOAD_BITS_PER_CHAR
CRC_BITS = 6
CONV_CONSTRAINT_LENGTH = 7
CONV_TAIL_BITS = CONV_CONSTRAINT_LENGTH - 1
CONV_INPUT_BITS = PAYLOAD_BITS + CRC_BITS + CONV_TAIL_BITS
CONV_RATE = 3
CONV_CODE_BITS = CONV_INPUT_BITS * CONV_RATE
CODE_BITS = 128
CONV_PADDING_BITS = CODE_BITS - CONV_CODE_BITS
CODE_BYTES = CODE_BITS // 8
assert CONV_CODE_BITS == 126 and CONV_PADDING_BITS == 2

TEMPORAL_SLOTS = 8
BITS_PER_SLOT = CODE_BITS // TEMPORAL_SLOTS
FRAME_SYMBOL_BITS = BITS_PER_SLOT + TEMPORAL_SLOTS
assert CODE_BITS % TEMPORAL_SLOTS == 0
assert CLIP_FRAMES % TEMPORAL_SLOTS == 0
TEMPORAL_REPETITIONS = CLIP_FRAMES // TEMPORAL_SLOTS

TARGET_TEXT = 'sabila'
CONTROL_TEXT = 'kontro'
CORE_NEURAL_QUALITY = 5
INTEGRITY_MAX_DISTANCE = 12

MAX_TRAIN_VIDEOS = 80
MAX_VAL_VIDEOS = 12
MAX_TEST_VIDEOS = 12

STEPS_PER_EPOCH = 50
VALIDATE_EVERY = 2
STAGE_PASS_PATIENCE = 2




OUTPUT_ROOT = Path('/content/drive/MyDrive/Video_data/output/v9_2_temporal_latent_validated')
MODEL_DIR = OUTPUT_ROOT / 'models'
BITSTREAM_DIR = OUTPUT_ROOT / 'bitstreams'
REPORT_DIR = OUTPUT_ROOT / 'reports' / 'v10_10'
for directory in (OUTPUT_ROOT, MODEL_DIR, BITSTREAM_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if RUN_MODE == 'train' and device.type != 'cuda':
    raise RuntimeError('Training V10.10 memerlukan GPU. Pilih runtime GPU Colab, lalu jalankan ulang bagian 1–8.')

KAGGLE_DATASET_SLUG = 'abdallahwagih/ucf101-videos'
DATASET_ROOT = Path(kagglehub.dataset_download(KAGGLE_DATASET_SLUG))
print('Dataset:', DATASET_ROOT)
print(
    f'Runtime profile={RUNTIME_PROFILE} | batch codec={CODEC_BATCH_SIZE} | '
    f'maksimum training per run={MAX_TRAINING_EPOCHS_PER_RUN} epoch/'
    f'{MAX_TRAINING_MINUTES_PER_RUN} menit'
)

# Matematika evaluasi sesuai protokol audit yang stabil pada Tesla T4.
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
print('Evaluation math: benchmark=False, deterministic=True, TF32=False')


In [ ]:
# 3. Discovery dan split anti-leakage
VIDEO_EXTENSIONS = {'.avi'}
all_video_paths = sorted(
    path for path in DATASET_ROOT.rglob('*.avi')
    if path.is_file()
)
if not all_video_paths:
    raise FileNotFoundError(f'Tidak ada video di {DATASET_ROOT}')

def class_name(path):
    match = re.match(r'^v_(.+?)_g\d+_c\d+$', path.stem)
    return match.group(1) if match else path.parent.name

def group_key(path):
    match = re.search(r'_g(\d+)_', path.stem)
    group = match.group(1) if match else path.stem
    return f'{class_name(path)}:{group}'

def has_part(path, expected):
    return expected.lower() in {part.lower() for part in path.relative_to(DATASET_ROOT).parts}

def group_split(paths, fractions=(0.70, 0.15, 0.15), seed=SEED):
    groups = sorted({group_key(p) for p in paths})
    rng = random.Random(seed)
    rng.shuffle(groups)
    n = len(groups)
    n_train = max(1, int(n * fractions[0]))
    n_val = max(1, int(n * fractions[1]))
    train_groups = set(groups[:n_train])
    val_groups = set(groups[n_train:n_train + n_val])
    test_groups = set(groups[n_train + n_val:])
    return (
        [p for p in paths if group_key(p) in train_groups],
        [p for p in paths if group_key(p) in val_groups],
        [p for p in paths if group_key(p) in test_groups],
    )

explicit_train = [p for p in all_video_paths if has_part(p, 'train')]
explicit_test = [p for p in all_video_paths if has_part(p, 'test')]

if not explicit_test:
    raise FileNotFoundError(
        f'Folder test UCF101 tidak ditemukan atau tidak berisi AVI di {DATASET_ROOT}'
    )

if explicit_train and explicit_test:
    # Test bawaan dataset tidak pernah dipakai untuk training atau pemilihan checkpoint.
    train_candidates, val_candidates, _ = group_split(
        explicit_train, fractions=(0.80, 0.20, 0.0)
    )
    test_candidates = explicit_test
    split_source = 'folder train/test dataset + validation berbasis group dari train'
else:
    train_candidates, val_candidates, test_candidates = group_split(all_video_paths)
    split_source = 'group-aware split 70/15/15'

def balanced_sample(paths, limit, seed):
    by_class = defaultdict(list)
    for path in paths:
        by_class[class_name(path)].append(path)
    rng = random.Random(seed)
    for values in by_class.values():
        rng.shuffle(values)
    chosen = []
    classes = sorted(by_class)
    while len(chosen) < min(limit, len(paths)):
        progressed = False
        for cls in classes:
            if by_class[cls] and len(chosen) < limit:
                chosen.append(by_class[cls].pop())
                progressed = True
        if not progressed:
            break
    return chosen

TRAIN_PATHS = balanced_sample(train_candidates, MAX_TRAIN_VIDEOS, SEED + 1)
VAL_PATHS = balanced_sample(val_candidates, MAX_VAL_VIDEOS, SEED + 2)
TEST_PATHS = balanced_sample(test_candidates, MAX_TEST_VIDEOS, SEED + 3)

train_set, val_set, test_set = map(set, (TRAIN_PATHS, VAL_PATHS, TEST_PATHS))
assert train_set.isdisjoint(val_set)
assert train_set.isdisjoint(test_set)
assert val_set.isdisjoint(test_set)
train_groups = {group_key(p) for p in TRAIN_PATHS}
val_groups = {group_key(p) for p in VAL_PATHS}
test_groups = {group_key(p) for p in TEST_PATHS}
assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)
assert all(has_part(path, 'test') and path.suffix.lower() == '.avi' for path in TEST_PATHS)

print(f'Ditemukan: {len(all_video_paths)} video AVI')
print('Sumber split:', split_source)
print(f'Train/Val/Test dipakai: {len(TRAIN_PATHS)}/{len(VAL_PATHS)}/{len(TEST_PATHS)}')
print('Final test: hanya AVI dari folder test; train tidak pernah masuk final test.')
print('Kelas train:', sorted({class_name(p) for p in TRAIN_PATHS}))
print('Leakage check: LULUS')


In [ ]:
# 4. I/O video dengan frame count, resolusi, dan durasi yang konsisten
def _resize_rgb(frame_bgr, frame_size=FRAME_SIZE):
    frame = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    frame = cv2.resize(frame, (frame_size[1], frame_size[0]), interpolation=cv2.INTER_AREA)
    return frame.astype(np.float32) / 255.0

def sample_clip(path, clip_frames=CLIP_FRAMES):
    cap = cv2.VideoCapture(str(path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        raise ValueError(f'Video tidak dapat dibaca: {path}')
    start = random.randint(0, max(total - clip_frames, 0))
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    frames = []
    for _ in range(clip_frames):
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(_resize_rgb(frame))
    cap.release()
    if not frames:
        raise ValueError(f'Tidak ada frame: {path}')
    while len(frames) < clip_frames:
        frames.append(frames[-1].copy())
    return np.stack(frames)

def load_eval_frames(path, max_frames=MAX_EVAL_FRAMES):
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        raise ValueError(f'Video tidak dapat dibaca: {path}')
    frames = []
    for _ in range(min(max_frames, total)):
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(_resize_rgb(frame))
    cap.release()
    if not frames:
        raise ValueError(f'Tidak ada frame evaluasi: {path}')
    return np.stack(frames), float(fps)

def frames_to_tensor(frames):
    return torch.from_numpy(np.asarray(frames)).permute(0, 3, 1, 2).float().to(device)

def tensor_to_frames(tensor):
    return tensor.detach().clamp(0, 1).permute(0, 2, 3, 1).cpu().numpy()

def read_encoded_video(path, expected_frames=None):
    """Decode via FFmpeg CLI agar AV1 terbaca saat OpenCV tidak punya decoder AV1."""
    path = Path(path)
    height, width = FRAME_SIZE
    cmd = [
        'ffmpeg', '-hide_banner', '-loglevel', 'error', '-i', str(path),
        '-an', '-vf', f'scale={width}:{height}:flags=area',
        '-f', 'rawvideo', '-pix_fmt', 'rgb24', 'pipe:1',
    ]
    result = subprocess.run(cmd, capture_output=True, timeout=120)
    bytes_per_frame = height * width * 3
    frame_count = len(result.stdout) // bytes_per_frame
    if result.returncode != 0 or frame_count == 0:
        detail = result.stderr.decode(errors='replace').strip()
        raise RuntimeError(f'FFmpeg gagal mendecode {path}: {detail}')
    usable = frame_count * bytes_per_frame
    frames = np.frombuffer(result.stdout[:usable], dtype=np.uint8).reshape(
        frame_count, height, width, 3
    ).astype(np.float32) / 255.0
    if expected_frames is not None:
        frames = frames[:expected_frames]
        if len(frames) < expected_frames:
            frames = np.concatenate([
                frames,
                np.repeat(frames[-1:], expected_frames - len(frames), axis=0),
            ], axis=0)
    return frames
def ffmpeg_encode(frames, output_path, fps, codec, crf):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    frames_u8 = np.clip(np.asarray(frames) * 255.0, 0, 255).astype(np.uint8)
    height, width = frames_u8.shape[1:3]
    cmd = [
        'ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
        '-f', 'rawvideo', '-pix_fmt', 'rgb24', '-s', f'{width}x{height}',
        '-r', str(fps), '-i', '-', '-an', '-c:v', codec,
    ]
    if codec == 'libaom-av1':
        cmd += ['-crf', str(crf), '-b:v', '0', '-cpu-used', '8', '-row-mt', '1']
    else:
        cmd += ['-crf', str(crf), '-preset', 'fast']
    cmd += ['-pix_fmt', 'yuv420p', str(output_path)]
    result = subprocess.run(cmd, input=frames_u8.tobytes(), capture_output=True, timeout=120)
    if result.returncode != 0 or not output_path.exists() or output_path.stat().st_size == 0:
        raise RuntimeError(result.stderr.decode(errors='replace'))
    return output_path
print('Video I/O siap, termasuk decode AV1 melalui FFmpeg CLI.')


In [ ]:
# 5. Payload convolutional FEC/CRC-6, temporal interleaving, dan extractor
CONV_GENERATORS = (0o171, 0o133, 0o165)
CONV_STATES = 1 << (CONV_CONSTRAINT_LENGTH - 1)
CONV_STATE_MASK = CONV_STATES - 1
CONV_REGISTER_MASK = (1 << CONV_CONSTRAINT_LENGTH) - 1

def _parity(value):
    return int(int(value).bit_count() & 1)

CONV_NEXT_STATE = np.zeros((CONV_STATES, 2), dtype=np.int16)
CONV_OUTPUT_BITS = np.zeros((CONV_STATES, 2, CONV_RATE), dtype=np.uint8)
for _state in range(CONV_STATES):
    for _input_bit in (0, 1):
        _register = ((_state << 1) | _input_bit) & CONV_REGISTER_MASK
        CONV_NEXT_STATE[_state, _input_bit] = _register & CONV_STATE_MASK
        CONV_OUTPUT_BITS[_state, _input_bit] = [
            _parity(_register & generator) for generator in CONV_GENERATORS
        ]
del _state, _input_bit, _register

def text_to_payload(text):
    text = str(text).lower()
    if len(text) != PAYLOAD_BYTES:
        raise ValueError(f'Teks harus tepat {PAYLOAD_BYTES} karakter')
    invalid = sorted(set(text) - set(PAYLOAD_ALPHABET))
    if invalid:
        raise ValueError(
            f'Karakter tidak didukung: {invalid}. Gunakan a-z atau 0-5.'
        )
    symbols = np.asarray(
        [PAYLOAD_ALPHABET.index(character) for character in text], dtype=np.uint8
    )
    bits = np.asarray([
        (int(symbol) >> shift) & 1
        for symbol in symbols for shift in range(PAYLOAD_BITS_PER_CHAR - 1, -1, -1)
    ], dtype=np.uint8)
    return torch.tensor(bits, dtype=torch.float32, device=device).unsqueeze(0)

def payload_to_text(payload_bits):
    bits = np.asarray(payload_bits, dtype=np.uint8).reshape(-1)[:PAYLOAD_BITS]
    symbols = bits.reshape(PAYLOAD_BYTES, PAYLOAD_BITS_PER_CHAR)
    weights = (1 << np.arange(PAYLOAD_BITS_PER_CHAR - 1, -1, -1)).astype(np.uint8)
    indices = symbols @ weights
    return ''.join(PAYLOAD_ALPHABET[int(index)] for index in indices)

def crc6(payload_bits):
    """CRC-6/CDMA2000-A atas 30 bit payload (polynomial 0x27)."""
    checksum = 0
    for payload_bit in np.asarray(payload_bits, dtype=np.uint8).reshape(-1):
        feedback = ((checksum >> 5) & 1) ^ int(payload_bit)
        checksum = (checksum << 1) & 0x3F
        if feedback:
            checksum ^= 0x27
    return np.asarray([
        (checksum >> shift) & 1 for shift in range(CRC_BITS - 1, -1, -1)
    ], dtype=np.uint8)

def convolutional_encode(message_bits):
    message = np.asarray(message_bits, dtype=np.uint8).reshape(-1)
    if message.size != PAYLOAD_BITS + CRC_BITS:
        raise ValueError('Convolutional encoder memerlukan 36 message bit.')
    terminated = np.concatenate([
        message, np.zeros(CONV_TAIL_BITS, dtype=np.uint8)
    ])
    state = 0
    encoded = np.zeros(CODE_BITS, dtype=np.uint8)
    cursor = 0
    for input_bit in terminated:
        bit = int(input_bit)
        encoded[cursor:cursor + CONV_RATE] = CONV_OUTPUT_BITS[state, bit]
        state = int(CONV_NEXT_STATE[state, bit])
        cursor += CONV_RATE
    if state != 0 or cursor != CONV_CODE_BITS:
        raise RuntimeError('Terminasi convolutional code gagal.')
    return encoded

def payload_to_codeword(payload):
    payload_np = payload.detach().round().to(torch.uint8).cpu().numpy()
    rows = []
    for row in payload_np:
        payload_bits = row[:PAYLOAD_BITS].astype(np.uint8)
        rows.append(convolutional_encode(np.concatenate([
            payload_bits, crc6(payload_bits)
        ])))
    return torch.tensor(np.stack(rows), dtype=torch.float32, device=device)

def viterbi_decode_soft(codeword_probabilities):
    probabilities = np.clip(
        np.asarray(codeword_probabilities, dtype=np.float64).reshape(-1)[:CONV_CODE_BITS]
        .reshape(-1, CONV_RATE),
        1e-6, 1.0 - 1e-6,
    )
    if probabilities.shape[0] != CONV_INPUT_BITS:
        raise ValueError(f'Viterbi memerlukan {CODE_BITS} code probabilities.')

    metrics = np.full(CONV_STATES, np.inf, dtype=np.float64)
    metrics[0] = 0.0
    parents = np.full(
        (CONV_INPUT_BITS, CONV_STATES), -1, dtype=np.int16
    )
    next_states = np.arange(CONV_STATES, dtype=np.int16)
    input_bits = (next_states & 1).astype(np.uint8)
    predecessor_zero = next_states >> 1
    predecessor_one = predecessor_zero | (CONV_STATES >> 1)

    for time_index, symbol_probability in enumerate(probabilities):
        log_probability = np.log(symbol_probability)
        log_inverse = np.log1p(-symbol_probability)
        branch_cost = -np.sum(
            CONV_OUTPUT_BITS * log_probability.reshape(1, 1, CONV_RATE) +
            (1 - CONV_OUTPUT_BITS) * log_inverse.reshape(1, 1, CONV_RATE),
            axis=2,
        )
        candidate_zero = (
            metrics[predecessor_zero] +
            branch_cost[predecessor_zero, input_bits]
        )
        candidate_one = (
            metrics[predecessor_one] +
            branch_cost[predecessor_one, input_bits]
        )
        choose_one = candidate_one < candidate_zero
        metrics = np.where(choose_one, candidate_one, candidate_zero)
        parents[time_index] = np.where(
            choose_one, predecessor_one, predecessor_zero
        )

    # Encoder selalu diterminasi dengan enam bit nol, jadi final state harus nol.
    state = 0
    decoded = np.empty(CONV_INPUT_BITS, dtype=np.uint8)
    for time_index in range(CONV_INPUT_BITS - 1, -1, -1):
        decoded[time_index] = state & 1
        state = int(parents[time_index, state])
    return decoded

def decode_codeword_soft(codeword_probabilities):
    probabilities = np.asarray(
        codeword_probabilities, dtype=np.float64
    ).reshape(-1)[:CODE_BITS]
    decoded = viterbi_decode_soft(probabilities)
    payload_bits = decoded[:PAYLOAD_BITS]
    checksum_bits = decoded[PAYLOAD_BITS:PAYLOAD_BITS + CRC_BITS]
    tail_bits = decoded[PAYLOAD_BITS + CRC_BITS:]
    crc_valid = bool(
        not np.any(tail_bits) and np.array_equal(checksum_bits, crc6(payload_bits))
    )
    decoded_message = decoded[:PAYLOAD_BITS + CRC_BITS]
    reconstructed_codeword = convolutional_encode(decoded_message)
    hard_codeword = (probabilities >= 0.5).astype(np.uint8)
    corrected_bits = int(np.sum(reconstructed_codeword != hard_codeword))
    return {
        'ecc_success': crc_valid,
        'crc_valid': crc_valid,
        'payload_bits': payload_bits,
        'raw_payload_bits': payload_bits.copy(),
        'decoded_text': (
            payload_to_text(payload_bits)
            if crc_valid else '<invalid-crc>'
        ),
        'raw_text': payload_to_text(payload_bits),
        'corrected_symbols': corrected_bits if crc_valid else np.nan,
        'soft_flips': corrected_bits if crc_valid else 0,
    }

def decode_codeword(codeword_bits):
    bits = np.asarray(codeword_bits, dtype=np.uint8).reshape(-1)[:CODE_BITS]
    probabilities = np.where(bits > 0, 1.0 - 1e-4, 1e-4)
    return decode_codeword_soft(probabilities)

def temporal_frame_targets(codeword, frames_per_payload):
    """Ubah [B,128] menjadi [B*T,16 bit + 8 slot-ID]."""
    batch = codeword.shape[0]
    slots = torch.arange(frames_per_payload, device=codeword.device) % TEMPORAL_SLOTS
    code_segments = codeword.view(batch, TEMPORAL_SLOTS, BITS_PER_SLOT)
    segments = code_segments[:, slots, :]
    slot_targets = slots.unsqueeze(0).expand(batch, -1)
    slot_one_hot = F.one_hot(slot_targets, TEMPORAL_SLOTS).float()
    symbols = torch.cat([segments, slot_one_hot], dim=-1)
    return (
        symbols.reshape(batch * frames_per_payload, FRAME_SYMBOL_BITS),
        segments.reshape(batch * frames_per_payload, BITS_PER_SLOT),
        slot_targets.reshape(batch * frames_per_payload),
    )

def infer_temporal_slots(slot_logits):
    """Infer phase siklik global; codec menjaga urutan frame meski kualitas turun."""
    slot_probabilities = torch.softmax(slot_logits, dim=1)
    frame_indices = torch.arange(slot_logits.shape[0], device=slot_logits.device)
    offset_scores = []
    for offset in range(TEMPORAL_SLOTS):
        expected = (frame_indices + offset) % TEMPORAL_SLOTS
        offset_scores.append(
            torch.log(slot_probabilities[frame_indices, expected].clamp_min(1e-8)).mean()
        )
    inferred_offset = int(torch.stack(offset_scores).argmax().item())
    aligned_slots = (frame_indices + inferred_offset) % TEMPORAL_SLOTS
    confidence = float(
        slot_probabilities[frame_indices, aligned_slots].mean().item()
    )
    return aligned_slots, inferred_offset, confidence, slot_probabilities

def aggregate_temporal_predictions(segment_logits, slot_logits, presence_logits):
    """Rakit codeword dengan CRC-aware cyclic phase search.

    Perubahan V10.3:
    - semua 8 kemungkinan phase temporal diuji;
    - frame diberi bobot berdasarkan slot-confidence DAN presence;
    - logits (bukan probability) yang dirata-ratakan agar evidence lemah tetapi
      konsisten tidak hilang;
    - bila ada kandidat dengan CRC valid, kandidat itu diprioritaskan.

    Ini penting pada H.264/H.265 CRF tinggi karena classifier slot dapat memiliki
    confidence rendah walaupun urutan frame codec sebenarnya tetap terjaga.
    """
    slot_probabilities = torch.softmax(slot_logits, dim=1)
    presence_probabilities = torch.sigmoid(presence_logits).reshape(-1)
    frame_indices = torch.arange(segment_logits.shape[0], device=segment_logits.device)

    candidates = []
    for offset in range(TEMPORAL_SLOTS):
        aligned_slots = (frame_indices + offset) % TEMPORAL_SLOTS
        aligned_slot_prob = slot_probabilities[frame_indices, aligned_slots]

        # Jangan biarkan satu frame ber-confidence sangat kecil menguasai rata-rata.
        frame_weights = (
            aligned_slot_prob.clamp_min(0.05) *
            presence_probabilities.clamp_min(0.10)
        ).to(segment_logits.dtype)

        assignment = F.one_hot(aligned_slots, TEMPORAL_SLOTS).to(
            dtype=segment_logits.dtype
        )
        weighted_assignment = assignment * frame_weights.unsqueeze(1)
        slot_mass = weighted_assignment.sum(dim=0).clamp_min(1e-6)

        assembled_logits = (
            weighted_assignment.transpose(0, 1) @ segment_logits
        ) / slot_mass.unsqueeze(1)

        code_probabilities = torch.sigmoid(assembled_logits).reshape(-1)
        code_bits = (code_probabilities >= 0.5).to(torch.uint8)

        # CRC+tail dari convolutional decoder memberikan sinyal phase yang jauh
        # lebih kuat daripada slot classifier sendirian.
        decoded = decode_codeword_soft(code_probabilities.detach().cpu().numpy())
        phase_log_likelihood = torch.log(
            aligned_slot_prob.clamp_min(1e-8)
        ).mean().item()
        bit_confidence = (
            (code_probabilities - 0.5).abs() * 2.0
        ).mean().item()

        candidates.append({
            'offset': offset,
            'aligned_slots': aligned_slots,
            'code_probabilities': code_probabilities,
            'code_bits': code_bits,
            'crc_valid': bool(decoded['crc_valid']),
            'phase_log_likelihood': float(phase_log_likelihood),
            'bit_confidence': float(bit_confidence),
            'slot_confidence': float(aligned_slot_prob.mean().item()),
        })

    crc_candidates = [c for c in candidates if c['crc_valid']]
    if crc_candidates:
        # CRC valid menjadi prioritas utama; jika lebih dari satu kandidat valid,
        # prioritaskan likelihood slot; confidence bit sebagai tie-break.
        best = max(
            crc_candidates,
            key=lambda c: (
                c['phase_log_likelihood'],
                c['bit_confidence'],
            ),
        )
    else:
        # Fallback aman ketika seluruh phase gagal CRC.
        best = max(
            candidates,
            key=lambda c: (
                c['phase_log_likelihood'],
                c['bit_confidence'],
            ),
        )

    slot_coverage = int(torch.unique(best['aligned_slots']).numel())
    presence = float(presence_probabilities.mean().item())

    return (
        best['code_bits'].cpu().numpy(),
        best['code_probabilities'].detach().cpu().numpy(),
        presence,
        slot_coverage,
        best['slot_confidence'],
    )

def group_count(channels):
    for groups in (8, 4, 2, 1):
        if channels % groups == 0:
            return groups
    return 1

class TemporalCarrierEmbedder(nn.Module):
    def __init__(
        self, latent_channels, latent_shape, symbol_bits=FRAME_SYMBOL_BITS,
        strength=1.20, carrier_scale=0.35,
    ):
        super().__init__()
        self.latent_channels = latent_channels
        self.latent_shape = tuple(latent_shape)
        self.symbol_bits = symbol_bits
        self.strength = strength
        self.carrier_scale = carrier_scale
        carriers = torch.randn(symbol_bits, latent_channels, *self.latent_shape)
        self.carriers = nn.Parameter(carriers)
        self.refine_gain = nn.Parameter(torch.tensor(0.10))
        self.refine = nn.Sequential(
            nn.Conv2d(latent_channels * 2, latent_channels, 3, padding=1),
            nn.GroupNorm(group_count(latent_channels), latent_channels), nn.SiLU(),
            nn.Conv2d(latent_channels, latent_channels, 3, padding=1),
            nn.GroupNorm(group_count(latent_channels), latent_channels), nn.SiLU(),
            nn.Conv2d(latent_channels, latent_channels, 3, padding=1),
        )

    def normalized_carriers(self):
        flat = self.carriers.flatten(1)
        scale = math.sqrt(flat.shape[1]) / flat.norm(dim=1, keepdim=True).clamp_min(1e-6)
        return (flat * scale).view_as(self.carriers)

    def forward(self, latent, frame_symbols):
        segment_symbols = frame_symbols[:, :BITS_PER_SLOT] * 2.0 - 1.0
        slot_one_hot = frame_symbols[:, BITS_PER_SLOT:]
        slot_symbols = slot_one_hot - (1.0 / TEMPORAL_SLOTS)
        centered_symbols = torch.cat([segment_symbols, slot_symbols], dim=1)
        carriers = self.normalized_carriers()
        carrier_field = torch.einsum('bs,schw->bchw', centered_symbols, carriers)
        carrier_field = carrier_field * (self.carrier_scale / math.sqrt(FRAME_SYMBOL_BITS))
        refinement = self.refine(torch.cat([latent, carrier_field], dim=1))
        gain = torch.clamp(self.refine_gain, 0.0, 0.50)
        delta = torch.tanh(carrier_field + gain * refinement)
        return latent + self.strength * delta, delta

class TemporalLatentExtractor(nn.Module):
    def __init__(self, latent_channels):
        super().__init__()
        hidden = max(192, latent_channels)
        self.features = nn.Sequential(
            nn.Conv2d(latent_channels, hidden, 3, padding=1),
            nn.GroupNorm(group_count(hidden), hidden), nn.SiLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.GroupNorm(group_count(hidden), hidden), nn.SiLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.GroupNorm(group_count(hidden), hidden), nn.SiLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.shared = nn.Sequential(
            nn.Flatten(), nn.Linear(hidden * 4 * 4, 512), nn.SiLU(), nn.Dropout(0.02)
        )
        self.segment_head = nn.Linear(512, BITS_PER_SLOT)
        self.slot_head = nn.Linear(512, TEMPORAL_SLOTS)
        self.presence_head = nn.Linear(512, 1)

    def encode_features(self, latent):
        return self.shared(self.features(latent))

    def classify(self, hidden):
        return (
            self.segment_head(hidden), self.slot_head(hidden),
            self.presence_head(hidden).squeeze(1),
        )

    def forward(self, latent):
        return self.classify(self.encode_features(latent))

target_payload = text_to_payload(TARGET_TEXT)
target_codeword = payload_to_codeword(target_payload)
assert decode_codeword(target_codeword.cpu().numpy())['decoded_text'] == TARGET_TEXT
_symbols, _segments, _slots = temporal_frame_targets(target_codeword, TEMPORAL_SLOTS)
_perfect_segment_logits = (_segments * 2.0 - 1.0) * 20.0
_perfect_slot_logits = F.one_hot(_slots, TEMPORAL_SLOTS).float() * 20.0
_assembled, _, _, _coverage, _ = aggregate_temporal_predictions(
    _perfect_segment_logits, _perfect_slot_logits,
    torch.full((TEMPORAL_SLOTS,), 20.0, device=device),
)
assert np.array_equal(
    _assembled, target_codeword.cpu().numpy().reshape(-1).astype(np.uint8)
)
assert _coverage == TEMPORAL_SLOTS
assert decode_codeword(_assembled)['decoded_text'] == TARGET_TEXT
del _symbols, _segments, _slots, _perfect_segment_logits, _perfect_slot_logits, _assembled
print(
    f'Payload {PAYLOAD_BITS} bit → convolutional FEC/CRC-6 {CODE_BITS} bit → '
    f'{TEMPORAL_SLOTS} slot × {BITS_PER_SLOT} bit.'
)


def aggregate_training_logits(segment_logits, payload_count, frames_per_payload):
    """Average corresponding slots across cycles; retain differentiability."""
    if frames_per_payload % TEMPORAL_SLOTS:
        raise ValueError('Clip harus merupakan kelipatan jumlah slot.')
    return segment_logits.reshape(
        payload_count, frames_per_payload // TEMPORAL_SLOTS,
        TEMPORAL_SLOTS, BITS_PER_SLOT,
    ).mean(dim=1).reshape(payload_count, CODE_BITS)


def temporal_preflight():
    # Runs on CPU before expensive training; no dataset or checkpoint needed.
    payload = torch.stack([text_to_payload(TARGET_TEXT).reshape(-1),
                           text_to_payload(CONTROL_TEXT).reshape(-1)]).cpu()
    code = payload_to_codeword(payload).cpu()
    for frames in (8, 16, 64):
        _, segments, slots = temporal_frame_targets(code, frames)
        logits = ((segments * 2 - 1) * 8).clone().requires_grad_(True)
        assembled = aggregate_training_logits(logits, 2, frames)
        assert assembled.shape == code.shape
        assert torch.equal((assembled > 0), code.bool())
        F.binary_cross_entropy_with_logits(assembled, code).backward()
        assert logits.grad is not None and torch.isfinite(logits.grad).all()
        assert (logits.grad.abs().sum(dim=1) > 0).all()
        for expected, probabilities in zip(payload, torch.sigmoid(assembled).detach().numpy()):
            decoded = decode_codeword_soft(probabilities)
            assert decoded['crc_valid'] and np.array_equal(decoded['payload_bits'], expected.numpy())
    print('PREFLIGHT LULUS: FEC, slot repetition 8/16/64 frame, batch 2, gradient ke semua frame.')

temporal_preflight()


In [ ]:
# 6. Neural codec backbone, latent embedding, dan entropy-coded bitstream
from compressai.zoo import bmshj2018_factorized

_neural_models = {}

def get_neural_model(quality):
    quality = int(quality)
    if quality not in _neural_models:
        model = bmshj2018_factorized(
            quality=quality, pretrained=True
        ).to(device).eval()
        model.update(force=True)
        for parameter in model.parameters():
            parameter.requires_grad_(False)
        _neural_models[quality] = model
    return _neural_models[quality]

CORE_CODEC = get_neural_model(CORE_NEURAL_QUALITY)
with torch.inference_mode():
    probe_latent = CORE_CODEC.g_a(torch.zeros(1, 3, *FRAME_SIZE, device=device))
LATENT_CHANNELS = int(probe_latent.shape[1])
LATENT_SHAPE = tuple(map(int, probe_latent.shape[-2:]))

latent_embedder = TemporalCarrierEmbedder(LATENT_CHANNELS, LATENT_SHAPE).to(device)
latent_extractor = TemporalLatentExtractor(LATENT_CHANNELS).to(device)

def quantize_latent_training(latent, noise=None):
    if noise is None:
        noise = torch.empty_like(latent).uniform_(-0.5, 0.5)
    return latent + noise, noise

def quantize_latent_eval(model, latent):
    medians = model.entropy_bottleneck._get_medians()
    return model.entropy_bottleneck.quantize(latent, 'dequantize', medians)

def latent_training_forward(frames, frame_symbols):
    latent = CORE_CODEC.g_a(frames)
    watermarked_latent, delta = latent_embedder(latent, frame_symbols)
    baseline_hat = quantize_latent_eval(CORE_CODEC, latent)
    quantized = quantize_latent_eval(CORE_CODEC, watermarked_latent)
    watermarked_hat = watermarked_latent + (quantized - watermarked_latent).detach()
    baseline_frames = CORE_CODEC.g_s(baseline_hat).clamp(0, 1)
    watermarked_frames = CORE_CODEC.g_s(watermarked_hat).clamp(0, 1)
    return baseline_frames, watermarked_frames, latent, watermarked_latent, delta

def latent_eval_forward(frames, frame_symbols):
    latent = CORE_CODEC.g_a(frames)
    watermarked_latent, delta = latent_embedder(latent, frame_symbols)
    baseline_hat = quantize_latent_eval(CORE_CODEC, latent)
    watermarked_hat = quantize_latent_eval(CORE_CODEC, watermarked_latent)
    baseline_frames = CORE_CODEC.g_s(baseline_hat).clamp(0, 1)
    watermarked_frames = CORE_CODEC.g_s(watermarked_hat).clamp(0, 1)
    return baseline_frames, watermarked_frames, latent, watermarked_latent, delta

def plain_neural_encode(frames, output_path, fps, quality):
    model = get_neural_model(quality)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    entries = []
    with torch.inference_mode():
        for start in range(0, len(frames), CODEC_BATCH_SIZE):
            x = frames_to_tensor(frames[start:start + CODEC_BATCH_SIZE])
            packed = model.compress(x)
            shape = tuple(map(int, packed['shape']))
            for batch_index in range(x.shape[0]):
                streams = [group[batch_index] for group in packed['strings']]
                entries.append((shape, streams))
    header = json.dumps({
        'codec': 'bmshj2018-factorized', 'quality': int(quality),
        'fps': float(fps), 'frames': len(entries),
        'height': int(frames.shape[1]), 'width': int(frames.shape[2]),
    }).encode('utf-8')
    with output_path.open('wb') as handle:
        handle.write(b'NVC1'); handle.write(struct.pack('<I', len(header))); handle.write(header)
        for shape, streams in entries:
            handle.write(struct.pack('<IIH', shape[0], shape[1], len(streams)))
            for stream in streams:
                handle.write(struct.pack('<I', len(stream))); handle.write(stream)
    return output_path

def plain_neural_decode(input_path):
    input_path = Path(input_path)
    with input_path.open('rb') as handle:
        if handle.read(4) != b'NVC1':
            raise ValueError('Bukan container NVC1')
        header_len = struct.unpack('<I', handle.read(4))[0]
        header = json.loads(handle.read(header_len).decode('utf-8'))
        model = get_neural_model(header['quality'])
        entries = []
        for _ in range(header['frames']):
            h, w, n_streams = struct.unpack('<IIH', handle.read(10))
            streams = []
            for _ in range(n_streams):
                size = struct.unpack('<I', handle.read(4))[0]
                streams.append(handle.read(size))
            entries.append(((h, w), streams))
    frames = []
    with torch.inference_mode():
        for start in range(0, len(entries), CODEC_BATCH_SIZE):
            batch = entries[start:start + CODEC_BATCH_SIZE]
            shape = batch[0][0]
            n_streams = len(batch[0][1])
            strings = [[entry[1][i] for entry in batch] for i in range(n_streams)]
            frames.extend(tensor_to_frames(model.decompress(strings, shape)['x_hat']))
    return np.stack(frames), header

def latent_watermark_encode(frames, output_path, fps, payload):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    codeword = payload_to_codeword(payload)
    frame_symbols, _, _ = temporal_frame_targets(codeword, len(frames))
    entries = []
    reconstructed = []
    with torch.inference_mode():
        for start in range(0, len(frames), CODEC_BATCH_SIZE):
            x = frames_to_tensor(frames[start:start + CODEC_BATCH_SIZE])
            symbols = frame_symbols[start:start + x.shape[0]]
            latent = CORE_CODEC.g_a(x)
            watermarked_latent, _ = latent_embedder(latent, symbols)
            strings = CORE_CODEC.entropy_bottleneck.compress(watermarked_latent)
            shape = tuple(map(int, watermarked_latent.shape[-2:]))
            latent_hat = CORE_CODEC.entropy_bottleneck.decompress(strings, shape)
            reconstructed.extend(tensor_to_frames(CORE_CODEC.g_s(latent_hat).clamp(0, 1)))
            entries.extend((shape, stream) for stream in strings)
    header = json.dumps({
        'codec': 'bmshj2018-factorized-latent-watermark',
        'quality': CORE_NEURAL_QUALITY, 'fps': float(fps),
        'frames': len(entries), 'height': int(frames.shape[1]),
        'width': int(frames.shape[2]), 'code_bits': CODE_BITS,
        'temporal_slots': TEMPORAL_SLOTS, 'bits_per_slot': BITS_PER_SLOT,
    }).encode('utf-8')
    with output_path.open('wb') as handle:
        handle.write(b'LWM1'); handle.write(struct.pack('<I', len(header))); handle.write(header)
        for shape, stream in entries:
            handle.write(struct.pack('<II', shape[0], shape[1]))
            handle.write(struct.pack('<I', len(stream))); handle.write(stream)
    return np.stack(reconstructed), output_path

def latent_watermark_decode(input_path):
    input_path = Path(input_path)
    with input_path.open('rb') as handle:
        if handle.read(4) != b'LWM1':
            raise ValueError('Bukan container latent watermark LWM1')
        header_len = struct.unpack('<I', handle.read(4))[0]
        header = json.loads(handle.read(header_len).decode('utf-8'))
        entries = []
        for _ in range(header['frames']):
            h, w = struct.unpack('<II', handle.read(8))
            size = struct.unpack('<I', handle.read(4))[0]
            entries.append(((h, w), handle.read(size)))
    frames = []
    with torch.inference_mode():
        for start in range(0, len(entries), CODEC_BATCH_SIZE):
            batch = entries[start:start + CODEC_BATCH_SIZE]
            shape = batch[0][0]
            strings = [entry[1] for entry in batch]
            latent_hat = CORE_CODEC.entropy_bottleneck.decompress(strings, shape)
            frames.extend(tensor_to_frames(CORE_CODEC.g_s(latent_hat).clamp(0, 1)))
    return np.stack(frames), header

print(
    f'Core neural quality={CORE_NEURAL_QUALITY}; latent={LATENT_CHANNELS}x'
    f'{LATENT_SHAPE[0]}x{LATENT_SHAPE[1]}; temporal carrier embedder siap.'
)


In [ ]:
# 7. Robustness attacks dan deterministic temporal curriculum V10.2
def ffmpeg_bpda(x, codec, crf, fps=25.0):
    with tempfile.TemporaryDirectory() as tmp:
        suffix = '.mkv' if codec == 'libaom-av1' else '.mp4'
        path = Path(tmp) / f'roundtrip{suffix}'
        frames = tensor_to_frames(x)
        ffmpeg_encode(frames, path, fps, codec, crf)
        reconstructed = frames_to_tensor(read_encoded_video(path, len(frames)))
    return x + (reconstructed - x).detach()

def neural_attack_differentiable(x, quality):
    model = get_neural_model(quality)
    model.eval()
    latent = model.g_a(x.clamp(0, 1))
    quantized = quantize_latent_eval(model, latent)
    latent_hat = latent + (quantized - latent).detach() if torch.is_grad_enabled() else quantized
    return model.g_s(latent_hat).clamp(0, 1)

def apply_attack(x, specification):
    name, level = specification
    if name == 'identity':
        return x
    if name == 'noise':
        noisy = x + torch.randn_like(x) * level
        quantized = torch.clamp(torch.round(noisy * 255) / 255, 0, 1)
        return x + (quantized - x).detach()
    if name == 'blur':
        return F.avg_pool2d(x, 3, stride=1, padding=1)
    if name == 'resize':
        reduced = F.interpolate(x, scale_factor=float(level), mode='bilinear', align_corners=False)
        return F.interpolate(reduced, size=x.shape[-2:], mode='bilinear', align_corners=False)
    if name == 'h264':
        return ffmpeg_bpda(x, 'libx264', int(level))
    if name == 'h265':
        return ffmpeg_bpda(x, 'libx265', int(level))
    if name == 'av1':
        return ffmpeg_bpda(x, 'libaom-av1', int(level))
    if name == 'neural':
        return neural_attack_differentiable(x, int(level))
    raise ValueError(specification)

def temporal_prediction_from_latent(latent):
    hidden = latent_extractor.encode_features(latent)
    segment_logits, slot_logits, presence_logits = latent_extractor.classify(hidden)
    return aggregate_temporal_predictions(segment_logits, slot_logits, presence_logits)

def temporal_prediction_tensor(frames):
    return temporal_prediction_from_latent(CORE_CODEC.g_a(frames))

def latent_eval_forward_batched(frames, frame_symbols, batch_size=CODEC_BATCH_SIZE):
    collected = [[] for _ in range(5)]
    with torch.inference_mode():
        for start in range(0, frames.shape[0], batch_size):
            stop = start + batch_size
            values = latent_eval_forward(frames[start:stop], frame_symbols[start:stop])
            for bucket, value in zip(collected, values):
                bucket.append(value)
    return tuple(torch.cat(values, dim=0) for values in collected)

TRAINING_STAGES = [{'name': 'progressive_crf_18',
  'max_epochs': 30,
  'min_epochs': 4,
  'extract_domain': 'reencoded_rgb',
  'lr': 0.0002,
  'embedder_lr': 3e-05,
  'freeze_embedder': False,
  'attacks': [('h264', 18), ('h265', 18), ('h264', 18), ('h265', 18), ('neural', 3)],
  'validation_attacks': [('h264', 18), ('h265', 18), ('neural', 3)],
  'payload_weight': 6.0,
  'logit_margin': 1.5,
  'margin_weight': 1.25,
  'auxiliary_weight': 1.0,
  'presence_weight': 0.4,
  'slot_weight': 0.8,
  'quality_target_mse': 0.00038,
  'quality_target_ssim': 0.94,
  'quality_weight': 3.0,
  'min_raw_code_bit_acc': 0.78,
  'min_post_ecc_exact': 0.8,
  'min_crc_valid': 0.8,
  'min_slot_accuracy': 0.8,
  'min_presence': 0.8,
  'min_negative': 0.95,
  'min_incremental_psnr': 34.0,
  'min_incremental_ssim': 0.94},
 {'name': 'progressive_crf_22',
  'max_epochs': 30,
  'min_epochs': 4,
  'extract_domain': 'reencoded_rgb',
  'lr': 0.0002,
  'embedder_lr': 3e-05,
  'freeze_embedder': False,
  'attacks': [('h264', 22),
              ('h265', 22),
              ('h264', 22),
              ('h265', 22),
              ('h264', 18),
              ('h265', 18),
              ('neural', 3)],
  'validation_attacks': [('h264', 18), ('h265', 18), ('h264', 22), ('h265', 22), ('neural', 3)],
  'payload_weight': 6.0,
  'logit_margin': 1.5,
  'margin_weight': 1.25,
  'auxiliary_weight': 1.0,
  'presence_weight': 0.4,
  'slot_weight': 0.8,
  'quality_target_mse': 0.00038,
  'quality_target_ssim': 0.94,
  'quality_weight': 3.0,
  'min_raw_code_bit_acc': 0.78,
  'min_post_ecc_exact': 0.8,
  'min_crc_valid': 0.8,
  'min_slot_accuracy': 0.8,
  'min_presence': 0.8,
  'min_negative': 0.95,
  'min_incremental_psnr': 34.0,
  'min_incremental_ssim': 0.94},
 {'name': 'progressive_crf_26',
  'max_epochs': 30,
  'min_epochs': 4,
  'extract_domain': 'reencoded_rgb',
  'lr': 0.0002,
  'embedder_lr': 3e-05,
  'freeze_embedder': False,
  'attacks': [('h264', 26),
              ('h265', 26),
              ('h264', 26),
              ('h265', 26),
              ('h264', 22),
              ('h265', 22),
              ('neural', 3)],
  'validation_attacks': [('h264', 22), ('h265', 22), ('h264', 26), ('h265', 26), ('neural', 3)],
  'payload_weight': 6.0,
  'logit_margin': 1.5,
  'margin_weight': 1.25,
  'auxiliary_weight': 1.0,
  'presence_weight': 0.4,
  'slot_weight': 0.8,
  'quality_target_mse': 0.00038,
  'quality_target_ssim': 0.94,
  'quality_weight': 3.0,
  'min_raw_code_bit_acc': 0.78,
  'min_post_ecc_exact': 0.8,
  'min_crc_valid': 0.8,
  'min_slot_accuracy': 0.8,
  'min_presence': 0.8,
  'min_negative': 0.95,
  'min_incremental_psnr': 34.0,
  'min_incremental_ssim': 0.94},
 {'name': 'progressive_crf_28',
  'max_epochs': 30,
  'min_epochs': 4,
  'extract_domain': 'reencoded_rgb',
  'lr': 0.0002,
  'embedder_lr': 3e-05,
  'freeze_embedder': False,
  'attacks': [('h264', 28),
              ('h265', 28),
              ('h264', 28),
              ('h265', 28),
              ('h264', 26),
              ('h265', 26),
              ('neural', 3)],
  'validation_attacks': [('h264', 26), ('h265', 26), ('h264', 28), ('h265', 28), ('neural', 3)],
  'payload_weight': 6.0,
  'logit_margin': 1.5,
  'margin_weight': 1.25,
  'auxiliary_weight': 1.0,
  'presence_weight': 0.4,
  'slot_weight': 0.8,
  'quality_target_mse': 0.00038,
  'quality_target_ssim': 0.94,
  'quality_weight': 3.0,
  'min_raw_code_bit_acc': 0.78,
  'min_post_ecc_exact': 0.8,
  'min_crc_valid': 0.8,
  'min_slot_accuracy': 0.8,
  'min_presence': 0.8,
  'min_negative': 0.95,
  'min_incremental_psnr': 34.0,
  'min_incremental_ssim': 0.94},
 {'name': 'progressive_crf_35',
  'max_epochs': 30,
  'min_epochs': 4,
  'extract_domain': 'reencoded_rgb',
  'lr': 0.0002,
  'embedder_lr': 3e-05,
  'freeze_embedder': False,
  'attacks': [('h264', 35),
              ('h265', 35),
              ('h264', 35),
              ('h265', 35),
              ('h264', 28),
              ('h265', 28),
              ('neural', 3)],
  'validation_attacks': [('h264', 28), ('h265', 28), ('h264', 35), ('h265', 35), ('neural', 3)],
  'payload_weight': 6.0,
  'logit_margin': 1.5,
  'margin_weight': 1.25,
  'auxiliary_weight': 1.0,
  'presence_weight': 0.4,
  'slot_weight': 0.8,
  'quality_target_mse': 0.00038,
  'quality_target_ssim': 0.94,
  'quality_weight': 3.0,
  'min_raw_code_bit_acc': 0.78,
  'min_post_ecc_exact': 0.8,
  'min_crc_valid': 0.8,
  'min_slot_accuracy': 0.8,
  'min_presence': 0.8,
  'min_negative': 0.95,
  'min_incremental_psnr': 34.0,
  'min_incremental_ssim': 0.94},
 {'name': 'progressive_crf_40',
  'max_epochs': 30,
  'min_epochs': 4,
  'extract_domain': 'reencoded_rgb',
  'lr': 0.0002,
  'embedder_lr': 3e-05,
  'freeze_embedder': False,
  'attacks': [('h264', 40),
              ('h265', 40),
              ('h264', 40),
              ('h265', 40),
              ('h264', 35),
              ('h265', 35),
              ('neural', 3),
              ('neural', 1)],
  'validation_attacks': [('h264', 35),
                         ('h265', 35),
                         ('h264', 40),
                         ('h265', 40),
                         ('neural', 3),
                         ('neural', 1)],
  'payload_weight': 6.0,
  'logit_margin': 1.5,
  'margin_weight': 1.25,
  'auxiliary_weight': 1.0,
  'presence_weight': 0.4,
  'slot_weight': 0.8,
  'quality_target_mse': 0.00038,
  'quality_target_ssim': 0.94,
  'quality_weight': 3.0,
  'min_raw_code_bit_acc': 0.78,
  'min_post_ecc_exact': 0.8,
  'min_crc_valid': 0.8,
  'min_slot_accuracy': 0.8,
  'min_presence': 0.8,
  'min_negative': 0.95,
  'min_incremental_psnr': 34.0,
  'min_incremental_ssim': 0.94}]

def stage_passed(stage, metrics):
    return (
        metrics['raw_code_bit_acc'] >= stage['min_raw_code_bit_acc'] and
        metrics['post_ecc_exact_recovery'] >= stage['min_post_ecc_exact'] and
        metrics['crc_valid_rate'] >= stage['min_crc_valid'] and
        metrics['slot_accuracy'] >= stage['min_slot_accuracy'] and
        metrics['presence_tpr'] >= stage['min_presence'] and
        metrics['negative_rejection'] >= stage['min_negative'] and
        metrics['incremental_watermark_psnr'] >= stage['min_incremental_psnr'] and
        metrics['incremental_watermark_ssim'] >= stage['min_incremental_ssim']
    )

VALIDATION_TEMPORAL_FRAMES = MAX_EVAL_FRAMES
STAGE_PRESENCE_THRESHOLD_GRID = np.linspace(0.50, 0.99, 50)

def deterministic_validation_paths(max_videos, validation_round=0):
    """Ambil subset validation yang dapat direproduksi dan bergilir."""
    paths = list(VAL_PATHS)
    if not paths:
        raise RuntimeError('VAL_PATHS kosong.')
    count = min(int(max_videos), len(paths))
    start = (int(validation_round) * count) % len(paths)
    return [paths[(start + offset) % len(paths)] for offset in range(count)]

def deterministic_validation_payload(path, validation_round=0):
    """Payload tetap per video/round agar perubahan metrik berasal dari model."""
    digest = hashlib.sha256(
        f'{SEED}|{Path(path).name}|{int(validation_round)}'.encode()
    ).digest()
    bits = np.unpackbits(np.frombuffer(digest, dtype=np.uint8), bitorder='big')
    bits = bits[:PAYLOAD_BITS].astype(np.float32)
    return torch.tensor(bits, device=device).unsqueeze(0)

def calibrated_stage_presence(positive_scores, negative_scores, min_negative):
    """Kalibrasi threshold tanpa mengubah target negative rejection stage."""
    positives = np.asarray(positive_scores, dtype=np.float64)
    negatives = np.asarray(negative_scores, dtype=np.float64)
    best = None
    for threshold in STAGE_PRESENCE_THRESHOLD_GRID:
        tpr = float(np.mean(positives >= threshold))
        rejection = float(np.mean(negatives < threshold))
        if rejection >= min_negative:
            candidate = (tpr, rejection, -float(threshold))
            if best is None or candidate > best[0]:
                best = (candidate, float(threshold), tpr, rejection)
    if best is None:
        threshold = 0.99
        return (
            threshold,
            float(np.mean(positives >= threshold)),
            float(np.mean(negatives < threshold)),
        )
    _, threshold, tpr, rejection = best
    return threshold, tpr, rejection

def quick_validation(stage, max_videos=4, validation_round=0, paths_override=None):
    latent_embedder.eval(); latent_extractor.eval()
    correct = total = exact = crc_count = samples = 0
    slot_correct = slot_total = 0
    frame_slot_correct = frame_slot_total = 0
    positive_scores, negative_scores, coverage_values = [], [], []
    incremental_psnr, incremental_ssim, overall_psnr = [], [], []
    paths = paths_override if paths_override is not None else deterministic_validation_paths(max_videos, validation_round)
    with torch.no_grad():
        for path in paths:
            frames_np, _ = load_eval_frames(
                path, max_frames=VALIDATION_TEMPORAL_FRAMES
            )
            usable_frames = (len(frames_np) // TEMPORAL_SLOTS) * TEMPORAL_SLOTS
            if usable_frames < TEMPORAL_SLOTS:
                raise RuntimeError(f'Frame validation tidak cukup: {path}')
            original = frames_to_tensor(frames_np[:usable_frames])
            payload = deterministic_validation_payload(path, validation_round)
            codeword = payload_to_codeword(payload)
            frame_symbols, _, slot_targets = temporal_frame_targets(
                codeword, original.shape[0]
            )
            baseline, watermarked, clean_latent, watermarked_latent, _ = latent_eval_forward_batched(
                original, frame_symbols
            )
            inc_mse = F.mse_loss(watermarked, baseline).item()
            incremental_psnr.append(-10.0 * math.log10(max(inc_mse, 1e-12)))
            incremental_ssim.append(differentiable_ssim(
                watermarked, baseline, data_range=1.0, size_average=True
            ).item())
            overall_mse = F.mse_loss(watermarked, original).item()
            overall_psnr.append(-10.0 * math.log10(max(overall_mse, 1e-12)))

            for attack in stage['validation_attacks']:
                if stage['extract_domain'] == 'quantized_latent':
                    positive_latent = quantize_latent_eval(CORE_CODEC, watermarked_latent)
                    negative_latent = quantize_latent_eval(CORE_CODEC, clean_latent)
                else:
                    positive_latent = CORE_CODEC.g_a(apply_attack(watermarked, attack))
                    negative_latent = CORE_CODEC.g_a(apply_attack(baseline, attack))

                positive_hidden = latent_extractor.encode_features(positive_latent)
                negative_hidden = latent_extractor.encode_features(negative_latent)
                segment_logits, slot_logits, presence_logits = latent_extractor.classify(
                    positive_hidden
                )
                _, _, negative_presence_logits = latent_extractor.classify(negative_hidden)
                predicted, probabilities, presence, slot_coverage, _ = aggregate_temporal_predictions(
                    segment_logits, slot_logits, presence_logits
                )
                outcome = decode_codeword_soft(probabilities)
                expected_code = codeword.cpu().numpy().reshape(-1).astype(np.uint8)
                target = payload.cpu().numpy().reshape(-1).astype(np.uint8)
                correct += int((predicted == expected_code).sum()); total += CODE_BITS
                exact_match = bool(
                    outcome['crc_valid'] and
                    np.array_equal(outcome['payload_bits'], target)
                )
                exact += int(exact_match)
                crc_count += int(outcome['crc_valid']); samples += 1
                aligned_slots, _, _, _ = infer_temporal_slots(slot_logits)
                slot_correct += int((aligned_slots == slot_targets).sum().item())
                slot_total += int(slot_targets.numel())
                frame_slot_correct += int(
                    (slot_logits.argmax(dim=1) == slot_targets).sum().item()
                )
                frame_slot_total += int(slot_targets.numel())
                # Presence score berasal dari model; label benar tidak menjadi input skor.
                positive_scores.append(float(presence))
                negative_scores.append(float(
                    torch.sigmoid(negative_presence_logits).mean().item()
                ))
                coverage_values.append(slot_coverage / TEMPORAL_SLOTS)

    threshold, presence_tpr, negative_rejection = calibrated_stage_presence(
        positive_scores, negative_scores, stage['min_negative']
    )
    latent_embedder.train(); latent_extractor.train()
    return {
        '_positive_scores': positive_scores, '_negative_scores': negative_scores,
        'raw_code_bit_acc': correct / max(total, 1),
        'post_ecc_exact_recovery': exact / max(samples, 1),
        'crc_valid_rate': crc_count / max(samples, 1),
        'slot_accuracy': slot_correct / max(slot_total, 1),
        'frame_slot_accuracy': frame_slot_correct / max(frame_slot_total, 1),
        'slot_coverage': float(np.mean(coverage_values)),
        'presence_threshold': threshold,
        'presence_tpr': presence_tpr,
        'negative_rejection': negative_rejection,
        'incremental_watermark_psnr': float(np.mean(incremental_psnr)),
        'incremental_watermark_ssim': float(np.mean(incremental_ssim)),
        'overall_original_to_watermarked_psnr': float(np.mean(overall_psnr)),
    }

# Percobaan kualitas CRF22; stage berikutnya belum diubah.
TRAINING_STAGES[1].update(lr=1e-4, embedder_lr=1e-5, quality_weight=6.0)

print(
    'Temporal curriculum V10.10:', [(s['name'], s['max_epochs']) for s in TRAINING_STAGES],
    f'validation_frames={VALIDATION_TEMPORAL_FRAMES}',
)


In [ ]:
def diagnostic_phase_candidates(segment_logits, slot_logits, presence_logits):
    """Rakit codeword dengan CRC-aware cyclic phase search.

    Perubahan V10.3:
    - semua 8 kemungkinan phase temporal diuji;
    - frame diberi bobot berdasarkan slot-confidence DAN presence;
    - logits (bukan probability) yang dirata-ratakan agar evidence lemah tetapi
      konsisten tidak hilang;
    - bila ada kandidat dengan CRC valid, kandidat itu diprioritaskan.

    Ini penting pada H.264/H.265 CRF tinggi karena classifier slot dapat memiliki
    confidence rendah walaupun urutan frame codec sebenarnya tetap terjaga.
    """
    slot_probabilities = torch.softmax(slot_logits, dim=1)
    presence_probabilities = torch.sigmoid(presence_logits).reshape(-1)
    frame_indices = torch.arange(segment_logits.shape[0], device=segment_logits.device)

    candidates = []
    for offset in range(TEMPORAL_SLOTS):
        aligned_slots = (frame_indices + offset) % TEMPORAL_SLOTS
        aligned_slot_prob = slot_probabilities[frame_indices, aligned_slots]

        # Jangan biarkan satu frame ber-confidence sangat kecil menguasai rata-rata.
        frame_weights = (
            aligned_slot_prob.clamp_min(0.05) *
            presence_probabilities.clamp_min(0.10)
        ).to(segment_logits.dtype)

        assignment = F.one_hot(aligned_slots, TEMPORAL_SLOTS).to(
            dtype=segment_logits.dtype
        )
        weighted_assignment = assignment * frame_weights.unsqueeze(1)
        slot_mass = weighted_assignment.sum(dim=0).clamp_min(1e-6)

        assembled_logits = (
            weighted_assignment.transpose(0, 1) @ segment_logits
        ) / slot_mass.unsqueeze(1)

        code_probabilities = torch.sigmoid(assembled_logits).reshape(-1)
        code_bits = (code_probabilities >= 0.5).to(torch.uint8)

        # CRC+tail dari convolutional decoder memberikan sinyal phase yang jauh
        # lebih kuat daripada slot classifier sendirian.
        decoded = decode_codeword_soft(code_probabilities.detach().cpu().numpy())
        phase_log_likelihood = torch.log(
            aligned_slot_prob.clamp_min(1e-8)
        ).mean().item()
        bit_confidence = (
            (code_probabilities - 0.5).abs() * 2.0
        ).mean().item()

        candidates.append({
            'offset': offset,
            'aligned_slots': aligned_slots,
            'code_probabilities': code_probabilities,
            'code_bits': code_bits,
            'crc_valid': bool(decoded['crc_valid']),
            'phase_log_likelihood': float(phase_log_likelihood),
            'bit_confidence': float(bit_confidence),
            'slot_confidence': float(aligned_slot_prob.mean().item()),
        })

    crc_candidates = [c for c in candidates if c['crc_valid']]
    if crc_candidates:
        # CRC valid menjadi prioritas utama; jika lebih dari satu kandidat valid,
        # prioritaskan likelihood slot; confidence bit sebagai tie-break.
        best = max(
            crc_candidates,
            key=lambda c: (
                c['phase_log_likelihood'],
                c['bit_confidence'],
            ),
        )
    else:
        # Fallback aman ketika seluruh phase gagal CRC.
        best = max(
            candidates,
            key=lambda c: (
                c['phase_log_likelihood'],
                c['bit_confidence'],
            ),
        )

    slot_coverage = int(torch.unique(best['aligned_slots']).numel())
    presence = float(presence_probabilities.mean().item())

    return candidates, best, presence
# Diagnostic only. No optimizer or training loop is instantiated.
DIAG_SCHEMA = 'stable_validation_v1011_v1'
DIAG_DIR = OUTPUT_ROOT / 'reports' / 'stable_eval_v1010_e864_vs_v1011_e868'
DIAG_DIR.mkdir(parents=True, exist_ok=True)
DIAG_CHECKPOINTS = {
    864: MODEL_DIR / 'best_v1010_stage2_faa0e6fc16.pth',
    868: MODEL_DIR / 'checkpoint_v1011_067bc9a7ae.pth',
}
DIAG_PATHS = deterministic_validation_paths(VALIDATION_VIDEOS_PER_CODEC, 0)
if len(DIAG_PATHS) != 6:
    raise ValueError('Diagnostic comparison requires the original six validation videos.')

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

fingerprints = {}
for epoch, path in DIAG_CHECKPOINTS.items():
    if not path.is_file():
        raise FileNotFoundError(f'Checkpoint diagnostic belum tersedia: {path}')
    saved = torch.load(path, map_location='cpu', weights_only=False)
    if (saved.get('global_epoch') != epoch or saved.get('stage_index') != (1 if epoch == 864 else 2)
            or saved.get('config_hash') != ('faa0e6fc16' if epoch == 864 else '067bc9a7ae')):
        raise ValueError(f'Identitas checkpoint salah: {path.name}; wajib epoch {epoch}, baseline V10.10 / hasil V10.11 sesuai epoch.')
    if epoch == 868 and saved.get('state', {}).get('global_epoch') != 868:
        raise ValueError('Checkpoint terakhir tidak memuat state epoch 868.')
    fingerprints[str(epoch)] = file_sha256(path)
    del saved

# Check the exact original split/experiment hash without executing training.
config_for_hash = {
    'version': '10.10-slot-likelihood-first-v1', 'seed': SEED,
    'frames': CLIP_FRAMES, 'payloads': PAYLOADS_PER_STEP,
    'frame_size': FRAME_SIZE, 'eval_frames': MAX_EVAL_FRAMES,
    'payload_bits': PAYLOAD_BITS, 'slots': TEMPORAL_SLOTS,
    'quality': CORE_NEURAL_QUALITY, 'stages': TRAINING_STAGES,
    'steps': STEPS_PER_EPOCH, 'validate_every': VALIDATE_EVERY,
    'validation_videos': VALIDATION_VIDEOS_PER_CODEC,
    'early_stop': [EARLY_STOP_ROUNDS, EARLY_STOP_MIN_DELTA],
    'pass_patience': STAGE_PASS_PATIENCE,
    'warm_start': WARM_START_CHECKPOINT,
    'decoder_policy': 'crc_then_slot_likelihood_then_bit_confidence',
    'strength_policy': 'fixed_at_warm_start',
    'splits': {name: [str(p.relative_to(DATASET_ROOT)) for p in paths]
               for name, paths in [('train', TRAIN_PATHS), ('val', VAL_PATHS), ('test', TEST_PATHS)]},
}
config_hash = hashlib.sha256(json.dumps(config_for_hash, sort_keys=True).encode()).hexdigest()[:10]
if config_hash != 'faa0e6fc16':
    raise ValueError('Konfigurasi/split harus sama dengan V10.10 faa0e6fc16.')

import platform, uuid
SESSION_ID = uuid.uuid4().hex
identity = dict(schema=DIAG_SCHEMA, checkpoints=fingerprints,
    videos=[str(p.relative_to(DATASET_ROOT)) for p in DIAG_PATHS],
    seed=SEED, validation_frames=VALIDATION_TEMPORAL_FRAMES,
    device=str(device), torch=str(torch.__version__), python=platform.python_version(),
    codec_batch_size=CODEC_BATCH_SIZE,
    math_policy=dict(benchmark=False,deterministic=True,cudnn_tf32=False,matmul_tf32=False),
    gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    cuda=torch.version.cuda,cudnn=torch.backends.cudnn.version(),
    ffmpeg=subprocess.run(['ffmpeg','-version'],capture_output=True,text=True,check=True).stdout,
    input_sha256={str(p.relative_to(DATASET_ROOT)):file_sha256(p) for p in DIAG_PATHS},
    checkpoint_labels={'864':'V10.10 baseline akhir CRF22','868':'V10.11 percobaan CRF26'},
    note='Two full-pipeline repeats per checkpoint, not 12 independent validation videos.')
job_hash = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:12]
JOB_DIR = DIAG_DIR / job_hash
JOB_DIR.mkdir(exist_ok=True)
(JOB_DIR / 'manifest.json').write_text(json.dumps(identity, indent=2))

def decode_diagnostic_candidate(candidate, target, expected):
    probs = candidate['code_probabilities'].detach().cpu().numpy()
    bits = (probs >= .5).astype(np.uint8)
    decoded = decode_codeword_soft(probs)
    exact_payload = bool(np.array_equal(decoded['payload_bits'], target))
    return dict(offset=int(candidate['offset']), crc_valid=bool(decoded['crc_valid']),
        exact_payload=exact_payload,
        post_ecc_exact_recovery=bool(decoded['crc_valid'] and exact_payload),
        crc_valid_wrong_payload=bool(decoded['crc_valid'] and not exact_payload),
        decoded_text=decoded['decoded_text'], raw_text=decoded['raw_text'],
        expected_text=payload_to_text(target),
        payload_bit_errors=int(np.count_nonzero(decoded['payload_bits'] != target)),
        raw_bit_errors=int(np.count_nonzero(bits != expected)),
        raw_BER=float(np.mean(bits != expected)),
        wrong_bit_indices=np.flatnonzero(bits != expected).tolist(),
        code_probabilities=probs.tolist(),
        bit_confidence=candidate['bit_confidence'],
        phase_log_likelihood=candidate['phase_log_likelihood'])

for epoch, checkpoint in DIAG_CHECKPOINTS.items():
    saved = torch.load(checkpoint, map_location=device, weights_only=False)
    latent_embedder.load_state_dict(saved['latent_embedder'], strict=True)
    latent_extractor.load_state_dict(saved['latent_extractor'], strict=True)
    latent_embedder.strength = float(saved['latent_strength'])
    latent_embedder.eval(); latent_extractor.eval(); CORE_CODEC.eval()
    del saved
    for index, path in enumerate(DIAG_PATHS * 2):
        destination = JOB_DIR / f'e{epoch}_video{index:02d}.json'
        if destination.exists():
            print('Resume: sudah selesai', destination.name, flush=True)
            continue
        seed_everything(SEED + index % len(DIAG_PATHS))
        print(f'Diagnosis epoch {epoch}: {index+1}/12 {path.name}', flush=True)
        with torch.no_grad():
            frames, _ = load_eval_frames(path, max_frames=VALIDATION_TEMPORAL_FRAMES)
            usable = len(frames) // TEMPORAL_SLOTS * TEMPORAL_SLOTS
            if usable < TEMPORAL_SLOTS:
                raise ValueError(f'Frame tidak cukup: {path}')
            original = frames_to_tensor(frames[:usable])
            payload = deterministic_validation_payload(path, 0)
            codeword = payload_to_codeword(payload)
            symbols, _, slot_targets = temporal_frame_targets(codeword, usable)
            baseline, watermarked, _, _, _ = latent_eval_forward_batched(original, symbols)
            psnr = -10 * math.log10(max(F.mse_loss(watermarked, baseline).item(), 1e-12))
            ssim_value = float(differentiable_ssim(watermarked, baseline, data_range=1.0, size_average=True))
            target = payload.cpu().numpy().reshape(-1).astype(np.uint8)
            expected = codeword.cpu().numpy().reshape(-1).astype(np.uint8)
            records = []
            for condition, attack in [('identity', ('identity', 0)),
                    ('h264_22', ('h264', 22)), ('h264_26', ('h264', 26)),
                    ('h265_22', ('h265', 22)), ('h265_26', ('h265', 26)),
                    ('neural_3', ('neural', 3))]:
                if index >= len(DIAG_PATHS): condition += '_repeat'
                attacked = apply_attack(watermarked, attack)
                hidden = latent_extractor.encode_features(CORE_CODEC.g_a(attacked))
                segments, slots, presence_logits = latent_extractor.classify(hidden)
                candidates, best, presence = diagnostic_phase_candidates(segments, slots, presence_logits)
                neg_hidden = latent_extractor.encode_features(CORE_CODEC.g_a(apply_attack(baseline, attack)))
                _, _, neg_logits = latent_extractor.classify(neg_hidden)
                negative_presence = float(torch.sigmoid(neg_logits).mean())
                # Ground truth only used after normal selection; never to choose normal decoder output.
                details = [decode_diagnostic_candidate(c, target, expected) for c in candidates]
                selected = next(c for c in details if c['offset'] == best['offset'])
                oracle = next(c for c in details if c['offset'] == 0)
                inferred, inferred_phase, _, _ = infer_temporal_slots(slots)
                records.append(dict(condition=condition, selected=selected, oracle=oracle,
                    phase_candidates=details, presence=presence, negative_presence=negative_presence,
                    inferred_phase=inferred_phase,
                    aligned_slot_accuracy=float((inferred == slot_targets).float().mean()),
                    frame_slot_accuracy=float((slots.argmax(1) == slot_targets).float().mean()),
                    incremental_watermark_psnr=psnr, incremental_watermark_ssim=ssim_value))
            report = dict(epoch=epoch, video=str(path.relative_to(DATASET_ROOT)),
                frames=usable, session_id=SESSION_ID, repeat=1+index//len(DIAG_PATHS), records=records)
        temp = destination.with_suffix('.tmp')
        temp.write_text(json.dumps(report, indent=2, allow_nan=False))
        os.replace(temp, destination)
        del original, baseline, watermarked, attacked, hidden, segments, slots, presence_logits, candidates, best
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

rows, phases = [], []
for epoch in DIAG_CHECKPOINTS:
    for index in range(len(DIAG_PATHS) * 2):
        report = json.loads((JOB_DIR / f'e{epoch}_video{index:02d}.json').read_text())
        for record in report['records']:
            selected, oracle = record['selected'], record['oracle']
            row = dict(epoch=epoch, video=report['video'], condition=record['condition'],
                session_id=report['session_id'],
                frames=report['frames'], selected_phase=selected['offset'],
                inferred_phase=record['inferred_phase'],
                oracle_exact=oracle['post_ecc_exact_recovery'], oracle_BER=oracle['raw_BER'],
                phase_rescue=not selected['post_ecc_exact_recovery'] and oracle['post_ecc_exact_recovery'],
                **{k:record[k] for k in ['presence','negative_presence','aligned_slot_accuracy','frame_slot_accuracy',
                    'incremental_watermark_psnr','incremental_watermark_ssim']},
                **{k:v for k,v in selected.items() if k not in ['offset','code_probabilities']})
            rows.append(row)
            for phase in record['phase_candidates']:
                phases.append(dict(epoch=epoch, video=report['video'],condition=record['condition'],
                    **{k:v for k,v in phase.items() if k != 'code_probabilities'}))
results = pd.DataFrame(rows)
results.to_csv(JOB_DIR / 'diagnostic_per_video.csv', index=False)
pd.DataFrame(phases).to_csv(JOB_DIR / 'diagnostic_phase_candidates.csv', index=False)
metrics = ['post_ecc_exact_recovery','raw_BER','crc_valid','crc_valid_wrong_payload',
    'oracle_exact','phase_rescue','incremental_watermark_psnr','incremental_watermark_ssim']
summary = results.groupby(['epoch','condition'])[metrics].mean().reset_index()
summary.to_csv(JOB_DIR / 'diagnostic_summary.csv', index=False)
first = results[~results.condition.str.endswith('_repeat')].copy()
second = results[results.condition.str.endswith('_repeat')].copy()
second['condition'] = second.condition.str.removesuffix('_repeat')
paired = first.merge(second,on=['epoch','video','condition'],suffixes=('_first','_repeat'))
paired['same_recovery'] = paired.post_ecc_exact_recovery_first == paired.post_ecc_exact_recovery_repeat
paired['same_raw_errors'] = paired.wrong_bit_indices_first == paired.wrong_bit_indices_repeat
paired['same_session'] = paired.session_id_first == paired.session_id_repeat
paired['same_psnr'] = paired.incremental_watermark_psnr_first == paired.incremental_watermark_psnr_repeat
paired[['epoch','video','condition','same_recovery','same_raw_errors','same_psnr','same_session']].to_csv(JOB_DIR/'repeat_consistency.csv',index=False)
comparison = results[(results.condition=='h264_26') & (results.epoch==864)].merge(
    results[(results.condition=='h264_26') & (results.epoch==868)],on='video',suffixes=('_864','_868'))
comparison['regressed'] = comparison.post_ecc_exact_recovery_864 & ~comparison.post_ecc_exact_recovery_868
comparison.to_csv(JOB_DIR / 'checkpoint_comparison.csv',index=False)
# Presence is calibrated across videos, never independently per video.
presence_rows=[]
for (epoch,condition),group in results.groupby(['epoch','condition']):
    threshold,tpr,rejection=calibrated_stage_presence(group.presence.tolist(),
        group.negative_presence.tolist(),TRAINING_STAGES[2]['min_negative'])
    presence_rows.append(dict(epoch=int(epoch),condition=condition,threshold=threshold,
        presence_tpr=tpr,negative_rejection=rejection))
pd.DataFrame(presence_rows).to_csv(JOB_DIR / 'presence_summary.csv',index=False)
# Evaluate the original stage gates separately for each pass; no averaging repeats into a larger sample.
stage=TRAINING_STAGES[2]
gates=dict(raw_code_bit_acc='min_raw_code_bit_acc',post_ecc_exact_recovery='min_post_ecc_exact',
    crc_valid_rate='min_crc_valid',slot_accuracy='min_slot_accuracy',presence_tpr='min_presence',
    negative_rejection='min_negative',incremental_watermark_psnr='min_incremental_psnr',
    incremental_watermark_ssim='min_incremental_ssim')
gate_rows=[];failures=[]
for (epoch,condition),group in results.groupby(['epoch','condition']):
    if condition.removesuffix('_repeat')=='identity':continue
    _,tpr,rejection=calibrated_stage_presence(group.presence.tolist(),group.negative_presence.tolist(),stage['min_negative'])
    m=dict(raw_code_bit_acc=1-float(group.raw_BER.mean()),
        post_ecc_exact_recovery=float(group.post_ecc_exact_recovery.mean()),
        crc_valid_rate=float(group.crc_valid.mean()),slot_accuracy=float(group.aligned_slot_accuracy.mean()),
        presence_tpr=tpr,negative_rejection=rejection,
        incremental_watermark_psnr=float(group.incremental_watermark_psnr.mean()),
        incremental_watermark_ssim=float(group.incremental_watermark_ssim.mean()))
    gate_rows.append(dict(epoch=int(epoch),condition=condition,videos=len(group),passed=bool(stage_passed(stage,m)),**m))
    for metric,key in gates.items():
        if m[metric]<stage[key]:failures.append(dict(epoch=int(epoch),condition=condition,metric=metric,value=m[metric],target=stage[key]))
pd.DataFrame(gate_rows).to_csv(JOB_DIR/'stage_gates.csv',index=False)
status=dict(training_executed=False,final_test_executed=False,
    repeated_recovery_consistent=bool(paired.same_recovery.all()),
    repeated_raw_errors_consistent=bool(paired.same_raw_errors.all()),
    repeated_psnr_consistent=bool(paired.same_psnr.all()),
    all_comparisons_same_session=bool(paired.same_session.all()),gate_failures=failures,
    notes='Controlled evaluation only. Does not establish cross-hardware determinism or advance training stages.')
(JOB_DIR/'evaluation_status.json').write_text(json.dumps(status,indent=2))
print(summary.to_string(index=False))
print('Video H264 CRF26 recovery turun 864 → 868:',comparison.loc[comparison.regressed,'video'].tolist())
print('Pengulangan konsisten:', bool(paired.same_recovery.all() and paired.same_raw_errors.all()))
print('Oracle hanya diagnosis. Hasil ini tidak meluluskan stage dan bukan final test.')
import zipfile
archive = DIAG_DIR / f'stable_eval_v1010_e864_vs_v1011_e868_{job_hash}.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in sorted(JOB_DIR.iterdir()):
        if path.suffix in ['.json','.csv']:z.write(path, path.name)
print('SELESAI. Kirim ZIP:', archive)
# Copy to Colab local storage for an accessible download link.
import shutil
from IPython.display import display, FileLink
local_archive = Path.cwd() / archive.name
if local_archive.resolve() != archive.resolve():shutil.copy2(archive, local_archive)
try:
    from google.colab import files
except ImportError:
    display(FileLink(str(local_archive)))
else:
    files.download(str(local_archive))
